## KNN с масштабированием

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from joblib import parallel_backend
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate

# Загрузка данных
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
X = train.drop(columns=["Id", "Survived"]).copy()
y = train["Survived"]
X_test = test.drop(columns="Id").copy()
for data in [X, X_test]:
    data["around_whealth"] = (data["Parch"] + data["SibSp"]) * data["Fare"]


## Подготовка признаков

In [2]:
# Преобразования внутри фолдов
numeric_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare", "around_whealth"]
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Пропуск")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_columns),
    ("categorical", categorical_transformer, ["Sex", "Embarked"]),
])
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier()),
])


## Подбор параметров

In [3]:
# Пятикратная кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = {"model__n_neighbors": [3, 5, 9, 15, 25], "model__weights": ["uniform", "distance"], "model__p": [1, 2]}
search = GridSearchCV(
    pipeline, param_grid, scoring={"f1": "f1", "roc_auc": "roc_auc"},
    refit="f1", cv=cv, n_jobs=4, return_train_score=True,
)
with parallel_backend("threading"):
    search.fit(X, y)
print("Лучшие параметры:", search.best_params_)
print("F1 при подборе:", round(search.best_score_, 4))
cv_results = pd.DataFrame(search.cv_results_)
display(cv_results[["params", "mean_train_f1", "mean_test_f1", "std_test_f1", "mean_test_roc_auc"]]
        .sort_values("mean_test_f1", ascending=False).head(10))


Лучшие параметры: {'model__n_neighbors': 25, 'model__p': 1, 'model__weights': 'distance'}
F1 при подборе: 0.7292


,params,mean_train_f1,mean_test_f1,std_test_f1,mean_test_roc_auc
17,"{'model__n_neighbors': 25, 'model__p': 1, 'mod...",0.976174,0.729216,0.032183,0.823888
12,"{'model__n_neighbors': 15, 'model__p': 1, 'mod...",0.750027,0.726205,0.047416,0.845660
19,"{'model__n_neighbors': 25, 'model__p': 2, 'mod...",0.976193,0.722774,0.039321,0.820109
13,"{'model__n_neighbors': 15, 'model__p': 1, 'mod...",0.976174,0.720702,0.028500,0.822364
9,"{'model__n_neighbors': 9, 'model__p': 1, 'mode...",0.976174,0.715521,0.026450,0.822268
4,"{'model__n_neighbors': 5, 'model__p': 1, 'mode...",0.805071,0.715106,0.025671,0.831649
15,"{'model__n_neighbors': 15, 'model__p': 2, 'mod...",0.976193,0.714345,0.036304,0.817950
16,"{'model__n_neighbors': 25, 'model__p': 1, 'mod...",0.737175,0.709671,0.042728,0.838548
6,"{'model__n_neighbors': 5, 'model__p': 2, 'mode...",0.800278,0.706819,0.023392,0.830696
2,"{'model__n_neighbors': 3, 'model__p': 2, 'mode...",0.829997,0.704940,0.026118,0.802327


## Независимая оценка подбора

In [4]:
# Вложенная кросс-валидация
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
with parallel_backend("threading"):
    nested_scores = cross_validate(
        search, X, y, cv=outer_cv, scoring={"f1": "f1", "roc_auc": "roc_auc"},
        n_jobs=1,
    )
fold_results = pd.DataFrame({
    "Фолд": range(1, 6),
    "F1": nested_scores["test_f1"],
    "ROC AUC": nested_scores["test_roc_auc"],
})
display(fold_results)
print("F1:", round(fold_results["F1"].mean(), 4), "±", round(fold_results["F1"].std(ddof=0), 4))
print("ROC AUC:", round(fold_results["ROC AUC"].mean(), 4), "±", round(fold_results["ROC AUC"].std(ddof=0), 4))


,Фолд,F1,ROC AUC
0,1,0.753846,0.865349
1,2,0.737589,0.852607
2,3,0.720588,0.840174
3,4,0.638298,0.763971
4,5,0.698413,0.822298


F1: 0.7097 ± 0.0402
ROC AUC: 0.8289 ± 0.0354


## Предсказания теста

In [5]:
# Сохранение посылки
best_model = search.best_estimator_
submission = pd.DataFrame({"Id": test["Id"], "Survived": best_model.predict(X_test)})
sample = pd.read_csv("data/sample_submission.csv")
assert submission.columns.tolist() == sample.columns.tolist()
assert submission["Id"].equals(sample["Id"])
assert submission["Survived"].isin([0, 1]).all()
assert not submission.isna().any().any()
submission.to_csv("submission_knn_scaled_cv.csv", index=False)
print("Сохранён submission_knn_scaled_cv.csv")


Сохранён submission_knn_scaled_cv.csv
